In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

torch.cuda.empty_cache()

In [2]:
from ast import literal_eval

df = pd.read_csv('data/matches_processed.csv')
df["Player"] = df["Player"].apply(literal_eval)

In [25]:
from torch.utils.data import Dataset
import numpy as np
import torch

class WinPredictionDataset(Dataset):
    def __init__(self, players, results):
        self.results = torch.tensor(results, dtype=torch.float32)
        
        # Preprocess all data first
        self.champion_ids = []
        self.positions = []
        self.ranks = []
        self.masteries = []
        self.team_ids = []
        
        for match in players:
            # Process each match
            champ_ids = []
            pos = []
            rks = []
            mast = []
            team = []
            mask = []
            
            for player in match:
                # Champion ID (ensure it's within embedding bounds)
                champ_ids.append(int(player[0]))
                
                # Position (convert -1 to [0,0,0,0,0] if needed)
                position = player[1] if isinstance(player[1], list) else [0]*5
                pos.append(position)
                
                # Rank (replace NaN with -1)
                rank = player[2] if not isinstance(player[2], str) else -1.0
                rks.append(rank)
                
                # Mastery (ensure float)
                mast.append(float(player[3]))
                
                # Team ID (convert to 0/1)
                team.append(int(player[4]))
                
                # Attention mask
                mask.append(0 if (isinstance(player[1], int) and player[1] == -1) else 1)
            
            # Convert to tensors
            self.champion_ids.append(torch.tensor(champ_ids, dtype=torch.long))
            self.positions.append(torch.tensor(pos, dtype=torch.float32))
            self.ranks.append(torch.tensor(rks, dtype=torch.float32).unsqueeze(-1))
            self.masteries.append(torch.tensor(mast, dtype=torch.float32).unsqueeze(-1))
            self.team_ids.append(torch.tensor(team, dtype=torch.long).unsqueeze(-1))

    
    def __len__(self):
        return len(self.champion_ids)
    
    def __getitem__(self, idx):
        return {
            "champion_ids": self.champion_ids[idx],
            "positions": self.positions[idx],
            "ranks": self.ranks[idx],
            "masteries": self.masteries[idx],
            "team_ids": self.team_ids[idx],
            "results": self.results[idx]
        }

In [39]:
import torch
import torch.nn as nn

class WinPredictionModel(nn.Module):
    def __init__(self, embedding_dim=32, dropout=0.5):
        super().__init__()
        # (1) Champion embedding and learnable features masks for pos and ranks
        self.champion_embedding = nn.Embedding(1000, embedding_dim)
        self.missing_pos = nn.Parameter(torch.randn(5))  # Learned placeholder
        self.missing_rank = nn.Parameter(torch.randn(1))  # Learned placeholder
        
        # (3) Pass the teams to a 2D CNN
        self.team_cnn = nn.Sequential(
            # Input shape: [batch, 1, 10, embedding_dim+5+1+1+1] (added channel dim)
            nn.Conv2d(1, 16, kernel_size=(3,3), padding=(1,1)),  # (players, features)
            nn.ReLU(),
            nn.MaxPool2d((2,2)),  # Downsamples to [3, feature_dim//2]
            
            nn.Conv2d(16, 32, kernel_size=(3,3), padding=(1,1)),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((3, 8))  # Fixed output [3,8]
        )
                
        # (4) Transformer encoder
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=int(32*(embedding_dim+5+1+1+1)/3), nhead=2, dropout=dropout, batch_first=True
            ),
            num_layers=1,
            enable_nested_tensor=True 
        )
        
        # (5) Final classifier
        self.head = nn.Sequential(
            nn.Linear(int(6*32*(embedding_dim+5+1+1+1)/3), embedding_dim),
            nn.Dropout(dropout),
            nn.ReLU(),
            nn.Linear(embedding_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, champion_ids, positions, ranks, masteries, team_ids):
        # (1) Pass through champion embedding
        champion_embedded = self.champion_embedding(champion_ids)  # [batch, 10, embedding_dim]
        
        pos_valid = (positions != -1).any(-1, keepdim=True)  # [batch,10,1]
        positions = torch.where(
            pos_valid.expand(-1,-1,5),
            positions,
            self.missing_pos.view(1,1,5)  # Learned placeholder
        )
        
        rank_valid = (ranks != -1).unsqueeze(-1)  # [batch,10,1]
        ranks = torch.where(
            rank_valid,
            ranks.unsqueeze(-1),
            self.missing_rank.view(1,1,1)  # Learned placeholder
        ).squeeze(-1)  # [batch,10,1]
        
        # (2) Concatenate all features
        x = torch.cat([
            champion_embedded,  # [batch, 10, embedding_dim]
            positions,  # [batch, 10, 5]
            ranks,  # [batch, 10, 1]
            masteries,  # [batch, 10, 1]
            team_ids   # [batch, 10, 1]
        ], dim=-1)  # [batch, 10, embedding_dim+5+1+1+1]
        
        # (3) Pass through CNN
        blue_team = x[:,:5,:].unsqueeze(1)  # [batch, 1, 5, embedding_dim+5+1+1+1]
        red_team = x[:,5:,:].unsqueeze(1)   # [batch, 1, 5, embedding_dim+5+1+1+1]
        
        blue_out = self.team_cnn(blue_team)  # [batch, 32, 3, (embedding_dim+5+1+1+1)/3]
        red_out = self.team_cnn(red_team)    # [batch, 32, 3, (embedding_dim+5+1+1+1)/3]
        
        x= torch.cat([blue_out, red_out], dim=2) # [batch, 32, 6, (embedding_dim+5+1+1+1)/3]
        x = x.permute(0, 2, 1, 3).flatten(2, 3)  # [batch, 6, 32*(embedding_dim+5+1+1+1)/3]
        
        # (4) Pass through transformer encoder
        x = self.encoder(x)  # [batch, 6, 32*(embedding_dim+5+1+1+1)/3]
        
         # (5) Flatten the playersand pass to classifier
        x = x.flatten(1) # [batch, 6* 32*(embedding_dim+5+1+1+1)/3]
        return self.head(x)

In [5]:
from sklearn.metrics import confusion_matrix, accuracy_score

def eval_model(model, val_loader, criterion, device):
    model.eval()
    
    all_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            champion_ids = batch["champion_ids"].to(device)
            positions = batch["positions"].to(device)
            ranks = batch["ranks"].to(device)
            masteries = batch["masteries"].to(device)
            team_ids = batch["team_ids"].to(device)            
            results = batch["results"].to(device)
            
            outputs = model(champion_ids, positions, ranks, masteries, team_ids)
            loss = criterion(outputs, results.unsqueeze(1)) 
            all_loss += loss.item()
            
            preds = np.round(outputs.cpu().numpy())
            
            all_preds.extend(preds.flatten().tolist())
            all_labels.extend(results.cpu().numpy().flatten().tolist())
            
    average_loss = all_loss / len(val_loader)            
    accuracy = accuracy_score(all_labels, all_preds)
    conf_matrix = confusion_matrix(all_labels, all_preds)  # Call the function from sklearn.metrics
    return average_loss, accuracy, conf_matrix

In [45]:
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
import torch
import numpy as np
import visualize

torch.manual_seed(42)
np.random.seed(42)

n_folds = 5
num_epochs = 20
learning_rate = 2e-4
learning_rate_decay = 0.1
weight_decay = 1e-3
embedding_dim = 16
dropout = 0.3
batch_size = 128

dataset = WinPredictionDataset(df["Player"].to_numpy(), df["Win"].to_numpy())

kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

fold_train_losses = []
fold_val_losses = []
fold_train_accuracies = []
fold_val_accuracies = []
fold_conf_matrices = []

for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    print(f"Fold {fold + 1}/{kf.n_splits}")
    
    trainset = torch.utils.data.Subset(dataset, train_idx)
    valset = torch.utils.data.Subset(dataset, val_idx)

    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(valset, batch_size=batch_size, shuffle=False)
    
    # Initialize model, optimizer, and loss function
    model = WinPredictionModel(embedding_dim=embedding_dim, dropout=dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=learning_rate_decay)
    criterion = nn.BCELoss()

    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    conf_matrices = []

    for epoch in range(num_epochs):
        train_loss = 0
        all_preds = []
        all_labels = []
        
        model.train()        
        for batch in train_loader:
            champion_ids = batch["champion_ids"].to(device)
            positions = batch["positions"].to(device)
            ranks = batch["ranks"].to(device)
            masteries = batch["masteries"].to(device)
            team_ids = batch["team_ids"].to(device)            
            results = batch["results"].to(device)
            
            optimizer.zero_grad()
            outputs = model(champion_ids, positions, ranks, masteries, team_ids)
            loss = criterion(outputs, results.unsqueeze(1))  # Ensure results are the same shape as outputs
            
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                train_loss += loss.item()
                
                preds = np.round(outputs.detach().cpu().numpy())
                
                all_preds.extend(preds.flatten().tolist())
                all_labels.extend(results.cpu().numpy().flatten().tolist())
            
        average_train_loss = train_loss / len(train_loader)
        train_accuracy = accuracy_score(all_labels, all_preds)
        average_val_loss, val_accuracy, conf_matrix = eval_model(model, val_loader, criterion, device)
        
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {average_train_loss}/{average_val_loss}, Accuracy: {train_accuracy}/{val_accuracy}")
        
        train_losses.append(average_train_loss)
        val_losses.append(average_val_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        
    fold_train_losses.append(train_losses)
    fold_val_losses.append(val_losses)
    fold_train_accuracies.append(train_accuracies)
    fold_val_accuracies.append(val_accuracies)
    fold_conf_matrices.append(conf_matrix)
    
# Average the results across folds
train_losses = np.mean(fold_train_losses, axis=0)
val_losses = np.mean(fold_val_losses, axis=0)
train_accuracies = np.mean(fold_train_accuracies, axis=0)
val_accuracies = np.mean(fold_val_accuracies, axis=0)
conf_matrices = np.mean(fold_conf_matrices, axis=0)

visualize.loss(train_losses, val_loss=val_losses, title=f"Train and Validation Losses across {n_folds} folds")
visualize.accuracy(train_acc=train_accuracies, val_acc=val_accuracies, title=f"Train and Validation Accuracy across {n_folds} folds")
visualize.confusion_matrix(conf_matrices, title=f"Confusion Matrix across {n_folds} folds")

Using device: cuda
Fold 1/5
Epoch 1/20, Loss: 0.6969673255133251/0.6937631145119667, Accuracy: 0.49825/0.4995
Epoch 2/20, Loss: 0.6970573881315807/0.6944438107311726, Accuracy: 0.494625/0.512
Epoch 3/20, Loss: 0.6924807334703112/0.6929190866649151, Accuracy: 0.51425/0.5135
Epoch 4/20, Loss: 0.6905887136383663/0.69245545566082, Accuracy: 0.523125/0.5045
Epoch 5/20, Loss: 0.6890126258607895/0.6925179436802864, Accuracy: 0.536625/0.5155
Epoch 6/20, Loss: 0.6881091282481239/0.6948060467839241, Accuracy: 0.53925/0.503
Epoch 7/20, Loss: 0.6828248548129249/0.6980259120464325, Accuracy: 0.556625/0.5035
Epoch 8/20, Loss: 0.6798552586918786/0.6941392049193382, Accuracy: 0.567375/0.512
Epoch 9/20, Loss: 0.6727852461830018/0.7044233307242393, Accuracy: 0.580875/0.5115
Epoch 10/20, Loss: 0.6615652905570136/0.6975600719451904, Accuracy: 0.602125/0.526
Epoch 11/20, Loss: 0.651930432471018/0.6986338384449482, Accuracy: 0.611625/0.5185
Epoch 12/20, Loss: 0.6389657457669576/0.7434392236173153, Accuracy:

KeyboardInterrupt: 